In [ ]:
import arcpy
import os
from datetime import datetime

from arcpy.sa import *
arcpy.CheckOutExtension("Spatial")

# bemenet (ahol a B04/B08 vannak)
root_folder = r"C:\Users\emese\Documents\ErasmusGIS\thesis\Data\vegetation\NDVI"

# kimenet (ÚJ mappa!)
output_folder = r"C:\Users\emese\Documents\ErasmusGIS\thesis\Data\vegetation\NDVI_clean"

arcpy.env.overwriteOutput = True

# output mappa létrehozása
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

for subdir, dirs, files in os.walk(root_folder):

    # csak valódi rasterek
    b04_files = [
        f for f in files 
        if "B04" in f and f.lower().endswith((".tif", ".tiff"))
    ]

    for b04 in b04_files:

        b04_path = os.path.join(subdir, b04)

        # pár keresése
        b08 = b04.replace("B04", "B08")
        b08_path = os.path.join(subdir, b08)

        if not os.path.exists(b08_path):
            print("❌ Hiányzik pár:", b08)
            continue

        try:
            # dátum kinyerése (első 10 karakter)
            date_str = b04[:10]
            datetime.strptime(date_str, "%Y-%m-%d")  # ellenőrzés

        except:
            print("❌ Hibás dátum formátum:", b04)
            continue

        print("✔ Feldolgozás:", b04)

        red = Raster(b04_path)
        nir = Raster(b08_path)

        # NDVI
        ndvi = (Float(nir) - Float(red)) / (Float(nir) + Float(red))

        # ÚJ TISZTA NÉV
        out_name = f"{date_str}_NDVI.tif"
        out_path = os.path.join(output_folder, out_name)

        ndvi.save(out_path)

        print("→ Mentve:", out_name)

print("\n🎉 NDVI újragenerálás kész.")